Importing required Libraries

In [1]:
import pandas as pd
import numpy as np
import tensorflow
from keras.src.callbacks import EarlyStopping
from tensorflow.keras.layers import Lambda, Embedding, Dot, Add, Input, Flatten
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
import faiss


Preprocessing Step
Includes
Converting non continuous values into continuous valuest
Dividing data into input and output by dropping not necessary columns
Splitting Training and Testing data

In [2]:
names = ['user_id', 'item_id', 'rating', 'timestamp']
data = pd.read_csv('u.data', sep='\t', header=None, names=names)
data = data.drop(columns=['timestamp'])
data['user_index'], user_mapping = pd.factorize(data['user_id'])
data['item_index'], item_mapping = pd.factorize(data['item_id'])
X = data.drop(columns=['user_id', 'item_id'])
Y = data['rating']

X_train, x_test, Y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=123)

In [3]:
user_id_input = Input(shape=(1, ), name='User_id_input')
item_id_input = Input(shape=(1, ), name='Item_id_input')

num_users = len(user_mapping)
num_items = len(item_mapping)

user_embed = Embedding(input_dim=num_users, output_dim=32, embeddings_regularizer=l2(1e-4), name="User_embedding_vector")(user_id_input)
item_embed = Embedding(input_dim=num_items, output_dim=32, embeddings_regularizer=l2(1e-4), name='Item_embedding_vector')(item_id_input)
user_embed_bias = Embedding(input_dim=num_users, output_dim=1, embeddings_regularizer=l2(1e-4), name="User_embedding_Bias")(user_id_input)
item_embed_bias = Embedding(input_dim=num_items, output_dim=1, embeddings_regularizer=l2(1e-4), name='Item_embedding_Bias')(item_id_input)

user_embed_flatten = Flatten(name='User_vector_flattened')(user_embed)
item_embed_flatten = Flatten(name='Item_vector_flattened')(item_embed)
user_flatten_bias = Flatten(name='User_bias_flattened')(user_embed_bias)
item_flatten_bias = Flatten(name='Item_bias_flattened')(item_embed_bias)

dot_product = Dot(axes=1, name='Dot_product')([user_embed_flatten, item_embed_flatten])
output_1 = Add(name='Addition_of_bias')([dot_product, user_flatten_bias, item_flatten_bias])

global_mean = np.mean(Y_train)
output = Lambda(lambda x:x+global_mean, name='Final_Output')(output_1)

model = Model([user_id_input, item_id_input], output)
model.compile(optimizer='adam', loss='mse')

In [4]:
x = [X_train['user_index'].values, X_train['item_index'].values]
y = Y_train.values
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model.fit(x, y, batch_size=256, epochs=15, validation_split=0.1, callbacks=[early_stopping])

Epoch 1/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 802us/step - loss: 1.2160 - val_loss: 1.1898
Epoch 2/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 457us/step - loss: 1.1127 - val_loss: 1.0907
Epoch 3/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 460us/step - loss: 1.0010 - val_loss: 1.0188
Epoch 4/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 555us/step - loss: 0.9317 - val_loss: 0.9941
Epoch 5/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 508us/step - loss: 0.8954 - val_loss: 0.9846
Epoch 6/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 481us/step - loss: 0.8717 - val_loss: 0.9809
Epoch 7/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 508us/step - loss: 0.8536 - val_loss: 0.9799
Epoch 8/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 516us/step - loss: 0.8383 - val_loss: 0.9813
Epoch 9/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 562us/step - loss: 0.8249 - val_loss: 0.9837
Epoch 10/15
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 523us/step - loss: 0.8121 - val_loss: 0.9872


In [5]:
user_weights = model.get_layer('User_embedding_vector').get_weights()[0]
item_weights = model.get_layer('Item_embedding_vector').get_weights()[0]
user_weights = user_weights.astype(np.float32)
item_weights = item_weights.astype(np.float32)
index = faiss.IndexFlatIP(32)
index.add(item_weights)


In [6]:
reshaped_user_10 = user_weights[10-1].reshape(1, 32)
distances, indices = index.search(reshaped_user_10, k=5)

In [7]:
print(f"The original movie id is {item_mapping[indices[0]]}")


The original movie id is Index([474, 127, 483, 511, 134], dtype='int64')
